In [ ]:
%matplotlib qt


Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


# bigP3BCI Study F：连续 EEG 人工质量复核

本 notebook 仅处理 `Train/CBCol`，每次运行一个 EDF cell。自动检测只生成候选，不修改 EDF、不设置 `raw.info['bads']`，也不生成 `BAD_` annotation。

候选颜色：大幅异常为红色、平直片段为橙色、疑似眨眼为蓝色、多类型重叠为紫色。混合 annotation 会直接写出组成，例如 `CANDIDATE_MIXED__large_amplitude+blink`。flash 事件由 `StimulusBegin` 上升沿定位，并由同一采样点的 `StimulusType` 分类：绿色竖线为 non-target（0），红色竖线为 target（1），灰色竖线表示意外取值。事件线只用于判断波形是否与刺激锁定，不属于异常 annotation。自动候选中的大幅异常也是红色，但表现为有宽度的 annotation 区间，可与 target 细竖线区分。MNE 1.12.1 支持 `annotation_colors`；若更换到不支持该参数的旧版本，仍可通过 `CANDIDATE_...` 名称和下方对照表识别。

> 路径相对于项目根目录，运行时把工作目录设为项目根目录。本 notebook 是人工复核记录，交互单元需要 Qt 图形界面。


In [2]:
from pathlib import Path
import importlib
import sys
import numpy as np
import pandas as pd
import mne

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not ((PROJECT_ROOT / 'README.md').exists() or (PROJECT_ROOT / '.git').exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not ((PROJECT_ROOT / 'README.md').exists() or (PROJECT_ROOT / '.git').exists()):
    raise FileNotFoundError('请在项目目录或子目录启动 Jupyter。')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import bigp3bci_continuous_qc as continuous_qc
importlib.reload(continuous_qc)

from bigp3bci_continuous_qc import (
    ANNOTATION_COLORS,
    DEFAULT_CONFIG,
    discover_train_cbcol,
    open_edf_for_manual_review,
    open_edf_for_manual_annotation,
    save_manual_annotation_session,
    scan_many_edfs,
)

STUDY_F_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF'
QC_ROOT = PROJECT_ROOT / 'outputs' / 'artifact_qc' / 'continuous_manual_qc'
FULL_OUTPUT = QC_ROOT / 'full'
PILOT_OUTPUT = QC_ROOT / 'pilot'
MANUAL_OUTPUT = QC_ROOT / 'manual'
TRAIN_FILES = discover_train_cbcol(STUDY_F_ROOT)

print('MNE:', mne.__version__)
print('Train/CBCol EDF files:', len(TRAIN_FILES))
print('Candidate annotation colors:', ANNOTATION_COLORS)
print('Automatic candidates are read-only references.')


MNE: 1.12.1
Train/CBCol EDF files: 90
Candidate annotation colors: {'CANDIDATE_large_amplitude': '#d62728', 'CANDIDATE_flat': '#ff7f0e', 'CANDIDATE_blink': '#1f77b4', 'CANDIDATE_MIXED': '#9467bd'}
Automatic candidates are read-only references.


## 自动筛查参数与重新计算

峰峰值、标准差和参考值均以 µV 计算。默认使用 2 秒窗口、1 秒步长、`robust z >= 6`；平直候选阈值为峰峰值不超过 0.5 µV 或标准差不超过 0.1 µV。`EEG_Fz` 的眨眼候选由 `find_eog_events` 产生，并使用 100 µV 阈值和 0.75 秒最小间隔以减少刺激节律造成的误报。

通常直接读取已经计算并验证过的表格。只有修改参数后，才把 `RECOMPUTE_ALL` 改为 `True`。

In [3]:
QC_CONFIG = {
    **DEFAULT_CONFIG,
    'window_sec': 2.0,
    'step_sec': 1.0,
    'large_robust_z': 6.0,
    'flat_ptp_uv': 0.5,
    'flat_std_uv': 0.1,
    'persistent_ratio': 0.20,
    'blink_threshold_uv': 100.0,
    'blink_min_separation_sec': 0.75,
    'merge_gap_sec': 0.25,
}

RECOMPUTE_ALL = False
if RECOMPUTE_ALL:
    candidate_details, merged_review, channel_summary, file_summary, annotation_verification = scan_many_edfs(
        TRAIN_FILES, FULL_OUTPUT, QC_CONFIG
    )
else:
    candidate_details = pd.read_csv(FULL_OUTPUT / 'candidate_details.csv')
    merged_review = pd.read_csv(FULL_OUTPUT / 'manual_review_candidates.csv', keep_default_na=False)
    channel_summary = pd.read_csv(FULL_OUTPUT / 'channel_summary.csv')
    file_summary = pd.read_csv(FULL_OUTPUT / 'file_summary.csv', keep_default_na=False)
    annotation_verification = pd.read_csv(FULL_OUTPUT / 'candidate_annotation_verification.csv')

INSTITUTION_BY_SUBJECT = {
    **{subject: 'Duke University' for subject in ['F_05', 'F_06', 'F_07', 'F_08']},
    **{subject: 'East Tennessee State University' for subject in ['F_03', 'F_20', 'F_21', 'F_23', 'F_24', 'F_25']},
}
file_summary.insert(0, 'institution', file_summary['subject'].map(INSTITUTION_BY_SUBJECT))

print('Files with candidates:', merged_review['file'].nunique())
print('Raw candidate details:', len(candidate_details))
print('Merged review segments:', len(merged_review))
print('All candidate annotations reload correctly:', bool(annotation_verification['reload_ok'].all()))
display(file_summary)


Files with candidates: 89
Raw candidate details: 4743
Merged review segments: 862
All candidate annotations reload correctly: True


,institution,subject,session,file,edf_path,n_large_amplitude_details,n_flat_details,n_blink_details,n_raw_candidate_details,n_merged_review_segments,persistent_candidate_channels,max_channel_anomaly_ratio
0,East Tennessee State University,F_03,SE001,F_03_SE001_CBCol_Train01.edf,...,9,0,0,9,2,,0.018433
1,East Tennessee State University,F_03,SE001,F_03_SE001_CBCol_Train02.edf,...,8,0,0,8,5,,0.013825
2,East Tennessee State University,F_03,SE001,F_03_SE001_CBCol_Train03.edf,...,47,0,0,47,4,,0.023041
3,East Tennessee State University,F_03,SE002,F_03_SE002_CBCol_Train01.edf,...,2,0,0,2,1,,0.009217
4,East Tennessee State University,F_03,SE002,F_03_SE002_CBCol_Train02.edf,...,36,0,0,36,1,,0.027650
...,...,...,...,...,...,...,...,...,...,...,...,...
85,East Tennessee State University,F_25,SE002,F_25_SE002_CBCol_Train02.edf,...,14,0,4,18,7,,0.032258
86,East Tennessee State University,F_25,SE002,F_25_SE002_CBCol_Train03.edf,...,3,0,7,10,9,,0.013825
87,East Tennessee State University,F_25,SE003,F_25_SE003_CBCol_Train01.edf,...,0,0,9,9,9,,0.000000
88,East Tennessee State University,F_25,SE003,F_25_SE003_CBCol_Train02.edf,...,0,0,7,7,5,,0.000000


## 自动候选与人工结果严格分离

上方自动候选表只作为只读参考，不在其中填写或保存人工结论。每个 EDF 的人工标注必须在其“纯人工标注窗口”中完成，并保存到独立的 `manual` 输出目录。绿色 non-target 与红色 target 事件线只用于判断波形是否与刺激锁定。

## 三个试点 EDF

请先运行下面三个 cell，确认 Qt 窗口可以滚动、16 个 EEG 通道均显示、annotation 与对照表一致。试点也按机构分组；每次只运行一个 cell，关闭前一个窗口后再运行下一个。

### Duke University 试点

#### 试点：`F_05_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [4]:
# F_05_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


Using matplotlib as 2D backend.
Flash markers: total=756, target=84, non-target=672, unknown=0 (display only)


,candidate_id,file,start_sec,end_sec,trigger_channels,candidate_types,annotation_description,detection_values,manual_review,manual_notes,reviewed_start_sec,reviewed_end_sec
22,F_05_SE001_CBCol_Train02_C0001,F_05_SE001_CBCol_Train02.edf,0.000000,4.729687,EEG_Fz;EEG_T7,CANDIDATE_blink;CANDIDATE_large_amplitude,CANDIDATE_MIXED__blink+large_amplitude,ptp=222.354 µV; reference=66.540 µV; robust_z=...,,,,
23,F_05_SE001_CBCol_Train02_C0002,F_05_SE001_CBCol_Train02.edf,5.000000,9.000000,EEG_C4;EEG_P3;EEG_PO7,CANDIDATE_large_amplitude,CANDIDATE_large_amplitude,ptp=125.340 µV; reference=54.432 µV; robust_z=...,,,,
24,F_05_SE001_CBCol_Train02_C0003,F_05_SE001_CBCol_Train02.edf,9.757812,11.057813,EEG_Fz,CANDIDATE_blink,CANDIDATE_blink,find_eog_events peak at 10.258 s; threshold=10...,,,,
25,F_05_SE001_CBCol_Train02_C0004,F_05_SE001_CBCol_Train02.edf,16.000000,25.342969,EEG_Fz;EEG_P3;EEG_T7,CANDIDATE_blink;CANDIDATE_large_amplitude,CANDIDATE_MIXED__blink+large_amplitude,ptp=233.586 µV; reference=66.540 µV; robust_z=...,,,,
26,F_05_SE001_CBCol_Train02_C0005,F_05_SE001_CBCol_Train02.edf,39.000000,42.000000,EEG_C4,CANDIDATE_large_amplitude,CANDIDATE_large_amplitude,ptp=137.664 µV; reference=54.432 µV; robust_z=...,,,,
27,F_05_SE001_CBCol_Train02_C0006,F_05_SE001_CBCol_Train02.edf,54.000000,57.000000,EEG_T7,CANDIDATE_large_amplitude,CANDIDATE_large_amplitude,ptp=181.122 µV; reference=66.540 µV; robust_z=...,,,,
28,F_05_SE001_CBCol_Train02_C0007,F_05_SE001_CBCol_Train02.edf,74.000000,77.000000,EEG_C4;EEG_Fz,CANDIDATE_blink;CANDIDATE_large_amplitude,CANDIDATE_MIXED__blink+large_amplitude,ptp=126.450 µV; reference=54.432 µV; robust_z=...,,,,
29,F_05_SE001_CBCol_Train02_C0008,F_05_SE001_CBCol_Train02.edf,121.000000,123.000000,EEG_P4,CANDIDATE_large_amplitude,CANDIDATE_large_amplitude,ptp=137.676 µV; reference=55.548 µV; robust_z=...,,,,
30,F_05_SE001_CBCol_Train02_C0009,F_05_SE001_CBCol_Train02.edf,141.000000,144.796094,EEG_C4;EEG_Fz,CANDIDATE_blink;CANDIDATE_large_amplitude,CANDIDATE_MIXED__blink+large_amplitude,ptp=125.712 µV; reference=54.432 µV; robust_z=...,,,,
31,F_05_SE001_CBCol_Train02_C0010,F_05_SE001_CBCol_Train02.edf,155.000000,158.000000,EEG_T7,CANDIDATE_large_amplitude,CANDIDATE_large_amplitude,ptp=262.548 µV; reference=66.540 µV; robust_z=...,,,,


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [5]:
# Manual-only window for F_05_SE001_CBCol_Train02.edf
manual_file_name = 'F_05_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE001_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [8]:
# Run after annotating F_05_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE001_CBCol_Train02-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_05_SE001_CBCol_Train02_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_05_SE001_CBCol_Train02_M0001,F_05_SE001_CBCol_Train02.edf,1.889938,2.193431,0.303494,MANUAL_bad_span,,,
1,F_05_SE001_CBCol_Train02_M0002,F_05_SE001_CBCol_Train02.edf,2.903883,3.248762,0.344879,MANUAL_bad_span,,,
2,F_05_SE001_CBCol_Train02_M0003,F_05_SE001_CBCol_Train02.edf,3.828159,4.221321,0.393162,MANUAL_bad_span,,,
3,F_05_SE001_CBCol_Train02_M0004,F_05_SE001_CBCol_Train02.edf,10.139447,10.470531,0.331084,MANUAL_bad_span,,,
4,F_05_SE001_CBCol_Train02_M0005,F_05_SE001_CBCol_Train02.edf,18.192551,18.499973,0.307422,MANUAL_bad_span,,,
5,F_05_SE001_CBCol_Train02_M0006,F_05_SE001_CBCol_Train02.edf,31.747137,32.095549,0.348412,MANUAL_bad_span,,,
6,F_05_SE001_CBCol_Train02_M0007,F_05_SE001_CBCol_Train02.edf,37.410534,37.854588,0.444054,MANUAL_bad_span,,,
7,F_05_SE001_CBCol_Train02_M0008,F_05_SE001_CBCol_Train02.edf,42.868982,43.121752,0.252769,MANUAL_bad_span,,,
8,F_05_SE001_CBCol_Train02_M0009,F_05_SE001_CBCol_Train02.edf,50.760781,50.974606,0.213825,MANUAL_bad_span,,,
9,F_05_SE001_CBCol_Train02_M0010,F_05_SE001_CBCol_Train02.edf,52.147195,52.512767,0.365572,MANUAL_bad_span,,,


### East Tennessee State University（ETSU）试点

#### 试点：`F_03_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [9]:
# Manual-only window for F_03_SE001_CBCol_Train01.edf
manual_file_name = 'F_03_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_03_SE001_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### 试点：`F_25_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [10]:
# Manual-only window for F_25_SE001_CBCol_Train01.edf
manual_file_name = 'F_25_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_25_SE001_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [11]:
# Run after annotating F_25_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_25_SE001_CBCol_Train01-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_25_SE001_CBCol_Train01_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_25_SE001_CBCol_Train01_M0001,F_25_SE001_CBCol_Train01.edf,1.034637,1.338131,0.303494,MANUAL_bad_span,,,
1,F_25_SE001_CBCol_Train01_M0002,F_25_SE001_CBCol_Train01.edf,3.028039,3.372918,0.344879,MANUAL_bad_span,,,
2,F_25_SE001_CBCol_Train01_M0003,F_25_SE001_CBCol_Train01.edf,11.284446,11.512066,0.227620,MANUAL_bad_span,,,


## 其余有候选的 Train/CBCol EDF

以下先按机构分组，再按 subject、session、文件名排序。三个试点文件不重复列出。每个 EDF 单独占一个可运行 cell。

### Duke University

#### `F_05` / `SE001` / `F_05_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_05_SE001_CBCol_Train03.edf
manual_file_name = 'F_05_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_05_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_05` / `SE001` / `F_05_SE001_CBCol_Train04.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE001_CBCol_Train04.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train04.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_05_SE001_CBCol_Train04.edf
manual_file_name = 'F_05_SE001_CBCol_Train04.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE001\\Train\\CBCol\\F_05_SE001_CBCol_Train04.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_05_SE001_CBCol_Train04.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE001_CBCol_Train04.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_05` / `SE002` / `F_05_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [12]:
# Manual-only window for F_05_SE002_CBCol_Train01.edf
manual_file_name = 'F_05_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE002_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [13]:
# Run after annotating F_05_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE002_CBCol_Train01-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_05_SE002_CBCol_Train01_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_05_SE002_CBCol_Train01_M0001,F_05_SE002_CBCol_Train01.edf,0.889788,1.020842,0.131054,MANUAL_bad_span,,,
1,F_05_SE002_CBCol_Train01_M0002,F_05_SE002_CBCol_Train01.edf,2.089968,2.358973,0.269006,MANUAL_bad_span,,,
2,F_05_SE002_CBCol_Train01_M0003,F_05_SE002_CBCol_Train01.edf,8.104660,8.449539,0.344879,MANUAL_bad_span,,,
3,F_05_SE002_CBCol_Train01_M0004,F_05_SE002_CBCol_Train01.edf,10.187730,10.346375,0.158644,MANUAL_bad_span,,,
4,F_05_SE002_CBCol_Train01_M0005,F_05_SE002_CBCol_Train01.edf,12.263903,12.601885,0.337982,MANUAL_bad_span,,,
5,F_05_SE002_CBCol_Train01_M0006,F_05_SE002_CBCol_Train01.edf,15.085014,15.340225,0.255211,MANUAL_bad_span,,,
6,F_05_SE002_CBCol_Train01_M0007,F_05_SE002_CBCol_Train01.edf,16.595585,16.912874,0.317289,MANUAL_bad_span,,,
7,F_05_SE002_CBCol_Train01_M0008,F_05_SE002_CBCol_Train01.edf,21.557819,21.764746,0.206927,MANUAL_bad_span,,,
8,F_05_SE002_CBCol_Train01_M0009,F_05_SE002_CBCol_Train01.edf,22.275168,22.647637,0.372469,MANUAL_bad_span,,,
9,F_05_SE002_CBCol_Train01_M0010,F_05_SE002_CBCol_Train01.edf,23.834021,24.178901,0.344879,MANUAL_bad_span,,,


#### `F_05` / `SE002` / `F_05_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [14]:
# Manual-only window for F_05_SE002_CBCol_Train02.edf
manual_file_name = 'F_05_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE002_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [15]:
# Run after annotating F_05_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE002_CBCol_Train02-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_05_SE002_CBCol_Train02_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_05_SE002_CBCol_Train02_M0001,F_05_SE002_CBCol_Train02.edf,2.772828,3.138400,0.365572,MANUAL_bad_span,,,
1,F_05_SE002_CBCol_Train02_M0002,F_05_SE002_CBCol_Train02.edf,7.345926,7.594239,0.248313,MANUAL_bad_span,,,
2,F_05_SE002_CBCol_Train02_M0003,F_05_SE002_CBCol_Train02.edf,8.463335,8.746136,0.282801,MANUAL_bad_span,,,
3,F_05_SE002_CBCol_Train02_M0004,F_05_SE002_CBCol_Train02.edf,11.043031,11.339627,0.296596,MANUAL_bad_span,,,
4,F_05_SE002_CBCol_Train02_M0005,F_05_SE002_CBCol_Train02.edf,13.022637,13.346824,0.324186,MANUAL_bad_span,,,
5,F_05_SE002_CBCol_Train02_M0006,F_05_SE002_CBCol_Train02.edf,16.050676,16.312784,0.262108,MANUAL_bad_span,,,
6,F_05_SE002_CBCol_Train02_M0007,F_05_SE002_CBCol_Train02.edf,17.519861,17.768174,0.248313,MANUAL_bad_span,,,
7,F_05_SE002_CBCol_Train02_M0008,F_05_SE002_CBCol_Train02.edf,25.131443,25.579786,0.448343,MANUAL_bad_span,,,
8,F_05_SE002_CBCol_Train02_M0009,F_05_SE002_CBCol_Train02.edf,27.069664,27.414544,0.344879,MANUAL_bad_span,,,
9,F_05_SE002_CBCol_Train02_M0010,F_05_SE002_CBCol_Train02.edf,28.835446,29.173427,0.337982,MANUAL_bad_span,,,


#### `F_05` / `SE002` / `F_05_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_05_SE002_CBCol_Train03.edf
manual_file_name = 'F_05_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE002\\Train\\CBCol\\F_05_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_05_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_05` / `SE003` / `F_05_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_05_SE003_CBCol_Train02.edf
manual_file_name = 'F_05_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_05_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_05` / `SE003` / `F_05_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_05_SE003_CBCol_Train03.edf
manual_file_name = 'F_05_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_05_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_05` / `SE003` / `F_05_SE003_CBCol_Train04.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_05_SE003_CBCol_Train04.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train04.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [33]:
# Manual-only window for F_05_SE003_CBCol_Train04.edf
manual_file_name = 'F_05_SE003_CBCol_Train04.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_05\\SE003\\Train\\CBCol\\F_05_SE003_CBCol_Train04.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE003_CBCol_Train04-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [34]:
# Run after annotating F_05_SE003_CBCol_Train04.edf in the manual-only Qt window
if manual_file_name != 'F_05_SE003_CBCol_Train04.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_05_SE003_CBCol_Train04-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_05_SE003_CBCol_Train04_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_05_SE003_CBCol_Train04_M0001,F_05_SE003_CBCol_Train04.edf,0.703655,1.113551,0.409896,MANUAL_bad_span,,,
1,F_05_SE003_CBCol_Train04_M0002,F_05_SE003_CBCol_Train04.edf,1.748890,1.987996,0.239106,MANUAL_bad_span,,,
2,F_05_SE003_CBCol_Train04_M0003,F_05_SE003_CBCol_Train04.edf,2.356902,2.739472,0.382570,MANUAL_bad_span,,,
3,F_05_SE003_CBCol_Train04_M0004,F_05_SE003_CBCol_Train04.edf,6.606157,6.899916,0.293759,MANUAL_bad_span,,,
4,F_05_SE003_CBCol_Train04_M0005,F_05_SE003_CBCol_Train04.edf,7.548918,7.904161,0.355243,MANUAL_bad_span,,,
5,F_05_SE003_CBCol_Train04_M0006,F_05_SE003_CBCol_Train04.edf,8.478016,9.010881,0.532865,MANUAL_bad_span,,,
6,F_05_SE003_CBCol_Train04_M0007,F_05_SE003_CBCol_Train04.edf,11.231151,11.572731,0.341580,MANUAL_bad_span,,,
7,F_05_SE003_CBCol_Train04_M0008,F_05_SE003_CBCol_Train04.edf,13.348947,13.793001,0.444054,MANUAL_bad_span,,,
8,F_05_SE003_CBCol_Train04_M0009,F_05_SE003_CBCol_Train04.edf,15.644364,16.088418,0.444054,MANUAL_bad_span,,,
9,F_05_SE003_CBCol_Train04_M0010,F_05_SE003_CBCol_Train04.edf,17.577707,17.850971,0.273264,MANUAL_bad_span,,,


#### `F_06` / `SE001` / `F_06_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [32]:
# Manual-only window for F_06_SE001_CBCol_Train01.edf
manual_file_name = 'F_06_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_06_SE001_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE001` / `F_06_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [31]:
# Manual-only window for F_06_SE001_CBCol_Train02.edf
manual_file_name = 'F_06_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_06_SE001_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE001` / `F_06_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [30]:
# Manual-only window for F_06_SE001_CBCol_Train03.edf
manual_file_name = 'F_06_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE001\\Train\\CBCol\\F_06_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_06_SE001_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE002` / `F_06_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [29]:
# Manual-only window for F_06_SE002_CBCol_Train01.edf
manual_file_name = 'F_06_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_06_SE002_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE002` / `F_06_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_06_SE002_CBCol_Train02.edf
manual_file_name = 'F_06_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE002` / `F_06_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_06_SE002_CBCol_Train03.edf
manual_file_name = 'F_06_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE002\\Train\\CBCol\\F_06_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE003` / `F_06_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_06_SE003_CBCol_Train01.edf
manual_file_name = 'F_06_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE003` / `F_06_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_06_SE003_CBCol_Train02.edf
manual_file_name = 'F_06_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_06` / `SE003` / `F_06_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_06_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_06_SE003_CBCol_Train03.edf
manual_file_name = 'F_06_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_06\\SE003\\Train\\CBCol\\F_06_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_06_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_06_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE001` / `F_07_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE001_CBCol_Train01.edf
manual_file_name = 'F_07_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE001` / `F_07_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE001_CBCol_Train03.edf
manual_file_name = 'F_07_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE001` / `F_07_SE001_CBCol_Train04.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE001_CBCol_Train04.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train04.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE001_CBCol_Train04.edf
manual_file_name = 'F_07_SE001_CBCol_Train04.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE001\\Train\\CBCol\\F_07_SE001_CBCol_Train04.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE001_CBCol_Train04.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE001_CBCol_Train04.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE002` / `F_07_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE002_CBCol_Train01.edf
manual_file_name = 'F_07_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE002` / `F_07_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE002_CBCol_Train02.edf
manual_file_name = 'F_07_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE002` / `F_07_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE002_CBCol_Train03.edf
manual_file_name = 'F_07_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE002\\Train\\CBCol\\F_07_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE003` / `F_07_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE003_CBCol_Train01.edf
manual_file_name = 'F_07_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE003` / `F_07_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE003_CBCol_Train02.edf
manual_file_name = 'F_07_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_07` / `SE003` / `F_07_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_07_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_07_SE003_CBCol_Train03.edf
manual_file_name = 'F_07_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_07\\SE003\\Train\\CBCol\\F_07_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_07_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_07_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE001` / `F_08_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE001_CBCol_Train01.edf
manual_file_name = 'F_08_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE001` / `F_08_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE001_CBCol_Train02.edf
manual_file_name = 'F_08_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE001` / `F_08_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE001_CBCol_Train03.edf
manual_file_name = 'F_08_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE001\\Train\\CBCol\\F_08_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE002` / `F_08_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE002_CBCol_Train01.edf
manual_file_name = 'F_08_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE002` / `F_08_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE002_CBCol_Train02.edf
manual_file_name = 'F_08_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE002` / `F_08_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE002_CBCol_Train03.edf
manual_file_name = 'F_08_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE002\\Train\\CBCol\\F_08_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE003` / `F_08_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE003_CBCol_Train01.edf
manual_file_name = 'F_08_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE003` / `F_08_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_08_SE003_CBCol_Train02.edf
manual_file_name = 'F_08_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_08_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_08` / `SE003` / `F_08_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_08_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [27]:
# Manual-only window for F_08_SE003_CBCol_Train03.edf
manual_file_name = 'F_08_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_08\\SE003\\Train\\CBCol\\F_08_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_08_SE003_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [28]:
# Run after annotating F_08_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_08_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


Normalized manual annotation labels:
  'BAD_' -> 'MANUAL_bad_span'
Saved manual annotations: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_08_SE003_CBCol_Train03-manual-annot.fif
Saved manual review table: outputs\artifact_qc\continuous_manual_qc\manual\manual_review_tables\F_08_SE003_CBCol_Train03_manual_review.csv


,manual_id,file,start_sec,end_sec,duration_sec,type,affected_channels,certainty,rationale
0,F_08_SE003_CBCol_Train03_M0001,F_08_SE003_CBCol_Train03.edf,0.806129,1.099888,0.293759,MANUAL_bad_span,,,
1,F_08_SE003_CBCol_Train03_M0002,F_08_SE003_CBCol_Train03.edf,2.315912,2.630166,0.314254,MANUAL_bad_span,,,
2,F_08_SE003_CBCol_Train03_M0003,F_08_SE003_CBCol_Train03.edf,3.395305,3.825696,0.430391,MANUAL_bad_span,,,
3,F_08_SE003_CBCol_Train03_M0004,F_08_SE003_CBCol_Train03.edf,16.498314,17.010684,0.512370,MANUAL_bad_span,,,
4,F_08_SE003_CBCol_Train03_M0005,F_08_SE003_CBCol_Train03.edf,26.382849,26.642449,0.259601,MANUAL_bad_span,,,
5,F_08_SE003_CBCol_Train03_M0006,F_08_SE003_CBCol_Train03.edf,35.865110,36.363816,0.498707,MANUAL_bad_span,,,
6,F_08_SE003_CBCol_Train03_M0007,F_08_SE003_CBCol_Train03.edf,38.365475,38.652402,0.286927,MANUAL_bad_span,,,
7,F_08_SE003_CBCol_Train03_M0008,F_08_SE003_CBCol_Train03.edf,49.719934,50.054682,0.334748,MANUAL_bad_span,,,
8,F_08_SE003_CBCol_Train03_M0009,F_08_SE003_CBCol_Train03.edf,58.355076,58.764972,0.409896,MANUAL_bad_span,,,
9,F_08_SE003_CBCol_Train03_M0010,F_08_SE003_CBCol_Train03.edf,65.211589,65.491684,0.280096,MANUAL_bad_span,,,


### East Tennessee State University（ETSU）

#### `F_03` / `SE001` / `F_03_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [26]:
# Manual-only window for F_03_SE001_CBCol_Train02.edf
manual_file_name = 'F_03_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_03_SE001_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE001` / `F_03_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [25]:
# Manual-only window for F_03_SE001_CBCol_Train03.edf
manual_file_name = 'F_03_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE001\\Train\\CBCol\\F_03_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_03_SE001_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE002` / `F_03_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE002_CBCol_Train01.edf
manual_file_name = 'F_03_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE002` / `F_03_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE002_CBCol_Train02.edf
manual_file_name = 'F_03_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE002` / `F_03_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE002_CBCol_Train03.edf
manual_file_name = 'F_03_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE002\\Train\\CBCol\\F_03_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE003` / `F_03_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE003_CBCol_Train01.edf
manual_file_name = 'F_03_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE003` / `F_03_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE003_CBCol_Train02.edf
manual_file_name = 'F_03_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_03` / `SE003` / `F_03_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_03_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_03_SE003_CBCol_Train03.edf
manual_file_name = 'F_03_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_03\\SE003\\Train\\CBCol\\F_03_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_03_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_03_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE001` / `F_20_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE001\\Train\\CBCol\\F_20_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_20_SE001_CBCol_Train02.edf
manual_file_name = 'F_20_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE001\\Train\\CBCol\\F_20_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE001` / `F_20_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE001\\Train\\CBCol\\F_20_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_20_SE001_CBCol_Train03.edf
manual_file_name = 'F_20_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE001\\Train\\CBCol\\F_20_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE002` / `F_20_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [24]:
# Manual-only window for F_20_SE002_CBCol_Train01.edf
manual_file_name = 'F_20_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=738, target=82, non-target=656, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_20_SE002_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE002` / `F_20_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [23]:
# Manual-only window for F_20_SE002_CBCol_Train02.edf
manual_file_name = 'F_20_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_20_SE002_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE002` / `F_20_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_20_SE002_CBCol_Train03.edf
manual_file_name = 'F_20_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE002\\Train\\CBCol\\F_20_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE003` / `F_20_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [22]:
# Manual-only window for F_20_SE003_CBCol_Train01.edf
manual_file_name = 'F_20_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_20_SE003_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE003` / `F_20_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_20_SE003_CBCol_Train02.edf
manual_file_name = 'F_20_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_20` / `SE003` / `F_20_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_20_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [21]:
# Manual-only window for F_20_SE003_CBCol_Train03.edf
manual_file_name = 'F_20_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_20\\SE003\\Train\\CBCol\\F_20_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_20_SE003_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_20_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_20_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE001` / `F_21_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE001_CBCol_Train01.edf
manual_file_name = 'F_21_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE001` / `F_21_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE001_CBCol_Train02.edf
manual_file_name = 'F_21_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE001` / `F_21_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [20]:
# Manual-only window for F_21_SE001_CBCol_Train03.edf
manual_file_name = 'F_21_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE001\\Train\\CBCol\\F_21_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_21_SE001_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE002` / `F_21_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [19]:
# Manual-only window for F_21_SE002_CBCol_Train01.edf
manual_file_name = 'F_21_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=738, target=82, non-target=656, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_21_SE002_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE002` / `F_21_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [18]:
# Manual-only window for F_21_SE002_CBCol_Train02.edf
manual_file_name = 'F_21_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_21_SE002_CBCol_Train02-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE002` / `F_21_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE002_CBCol_Train03.edf
manual_file_name = 'F_21_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE002\\Train\\CBCol\\F_21_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE003` / `F_21_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE003_CBCol_Train01.edf
manual_file_name = 'F_21_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE003` / `F_21_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE003_CBCol_Train02.edf
manual_file_name = 'F_21_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_21` / `SE003` / `F_21_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_21_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_21_SE003_CBCol_Train03.edf
manual_file_name = 'F_21_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_21\\SE003\\Train\\CBCol\\F_21_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_21_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_21_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE001` / `F_23_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE001_CBCol_Train02.edf
manual_file_name = 'F_23_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE001` / `F_23_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE001_CBCol_Train03.edf
manual_file_name = 'F_23_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE001` / `F_23_SE001_CBCol_Train05.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE001_CBCol_Train05.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train05.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE001_CBCol_Train05.edf
manual_file_name = 'F_23_SE001_CBCol_Train05.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE001\\Train\\CBCol\\F_23_SE001_CBCol_Train05.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE001_CBCol_Train05.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE001_CBCol_Train05.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE002` / `F_23_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE002_CBCol_Train02.edf
manual_file_name = 'F_23_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE002` / `F_23_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE002_CBCol_Train03.edf
manual_file_name = 'F_23_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE002` / `F_23_SE002_CBCol_Train04.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE002_CBCol_Train04.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train04.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE002_CBCol_Train04.edf
manual_file_name = 'F_23_SE002_CBCol_Train04.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE002\\Train\\CBCol\\F_23_SE002_CBCol_Train04.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE002_CBCol_Train04.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE002_CBCol_Train04.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE003` / `F_23_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE003_CBCol_Train01.edf
manual_file_name = 'F_23_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE003` / `F_23_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE003_CBCol_Train02.edf
manual_file_name = 'F_23_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_23` / `SE003` / `F_23_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_23_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_23_SE003_CBCol_Train03.edf
manual_file_name = 'F_23_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_23\\SE003\\Train\\CBCol\\F_23_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_23_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_23_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE001` / `F_24_SE001_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE001_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE001_CBCol_Train01.edf
manual_file_name = 'F_24_SE001_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE001_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE001_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE001` / `F_24_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE001_CBCol_Train02.edf
manual_file_name = 'F_24_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE001` / `F_24_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE001_CBCol_Train03.edf
manual_file_name = 'F_24_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE001\\Train\\CBCol\\F_24_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE002` / `F_24_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE002_CBCol_Train01.edf
manual_file_name = 'F_24_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE002` / `F_24_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE002_CBCol_Train02.edf
manual_file_name = 'F_24_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE002` / `F_24_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE002_CBCol_Train03.edf
manual_file_name = 'F_24_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE002\\Train\\CBCol\\F_24_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE003` / `F_24_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE003_CBCol_Train01.edf
manual_file_name = 'F_24_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE003` / `F_24_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE003_CBCol_Train02.edf
manual_file_name = 'F_24_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_24` / `SE003` / `F_24_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_24_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_24_SE003_CBCol_Train03.edf
manual_file_name = 'F_24_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_24\\SE003\\Train\\CBCol\\F_24_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_24_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_24_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE001` / `F_25_SE001_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE001_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE001_CBCol_Train02.edf
manual_file_name = 'F_25_SE001_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE001_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE001_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE001` / `F_25_SE001_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE001_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE001_CBCol_Train03.edf
manual_file_name = 'F_25_SE001_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE001\\Train\\CBCol\\F_25_SE001_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE001_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE001_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE002` / `F_25_SE002_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE002_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE002_CBCol_Train01.edf
manual_file_name = 'F_25_SE002_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE002_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE002_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE002` / `F_25_SE002_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE002_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE002_CBCol_Train02.edf
manual_file_name = 'F_25_SE002_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE002_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE002_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE002` / `F_25_SE002_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE002_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE002_CBCol_Train03.edf
manual_file_name = 'F_25_SE002_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE002\\Train\\CBCol\\F_25_SE002_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE002_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE002_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE003` / `F_25_SE003_CBCol_Train01.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE003_CBCol_Train01.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train01.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [16]:
# Manual-only window for F_25_SE003_CBCol_Train01.edf
manual_file_name = 'F_25_SE003_CBCol_Train01.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train01.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_25_SE003_CBCol_Train01-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE003_CBCol_Train01.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE003_CBCol_Train01.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE003` / `F_25_SE003_CBCol_Train02.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE003_CBCol_Train02.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train02.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [ ]:
# Manual-only window for F_25_SE003_CBCol_Train02.edf
manual_file_name = 'F_25_SE003_CBCol_Train02.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train02.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE003_CBCol_Train02.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE003_CBCol_Train02.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)


#### `F_25` / `SE003` / `F_25_SE003_CBCol_Train03.edf`

##### 自动候选窗口与候选对照表

In [ ]:
# F_25_SE003_CBCol_Train03.edf
review_raw, review_browser, review_table = open_edf_for_manual_review(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train03.edf'), merged_review, duration=20.0
)
display(review_table)


##### 纯人工标注窗口（不加载自动候选）

此 cell 会重新读取 EDF，只加载此前保存的 `MANUAL_...` annotation，并保留分类型的 flash 事件线：绿色为 non-target，红色为 target，灰色为无法识别的 `StimulusType`。按 `a` 进入 MNE annotation 模式。建议人工描述以 `MANUAL_` 开头；若使用 MNE 默认的 `BAD_` 或普通名称，保存时会自动转换成 `MANUAL_...`。候选表只作为上方参考，不会进入这份 raw。

In [17]:
# Manual-only window for F_25_SE003_CBCol_Train03.edf
manual_file_name = 'F_25_SE003_CBCol_Train03.edf'
manual_raw, manual_browser, manual_annotation_file = open_edf_for_manual_annotation(
    Path('data\\raw\\bigP3BCI-data\\StudyF\\F_25\\SE003\\Train\\CBCol\\F_25_SE003_CBCol_Train03.edf'), MANUAL_OUTPUT, duration=20.0
)
print('Manual annotation file:', manual_annotation_file)


Manual-only window: candidate annotations=0; loaded manual annotations=0; flash markers total=756, target=84, non-target=672, unknown=0
Use annotation descriptions beginning with MANUAL_ before saving.
Manual annotation file: outputs\artifact_qc\continuous_manual_qc\manual\manual_annotations\F_25_SE003_CBCol_Train03-manual-annot.fif


##### 保存该 EDF 的人工 annotation 与复核表

完成当前文件标注后运行。它保存 `MANUAL_...` annotation，并自动规范化 MNE 默认的 `BAD_...` 名称；复核表中的 `affected_channels`、`certainty`、`rationale` 可随后填写。重新保存相同区间时会保留已有文字记录。

In [ ]:
# Run after annotating F_25_SE003_CBCol_Train03.edf in the manual-only Qt window
if manual_file_name != 'F_25_SE003_CBCol_Train03.edf':
    raise RuntimeError('manual_raw belongs to another EDF; rerun the manual-only cell above first.')
manual_annotation_file, manual_review_file, manual_review_table = save_manual_annotation_session(
    manual_raw, manual_file_name, MANUAL_OUTPUT
)
display(manual_review_table)
